# 01 — REF concepts

This notebook introduces the core vocabulary of the **Rapid Evaluation Framework (REF)**.
By the end you will know what a *diagnostic*, *provider*, *execution*, *metric* and *dataset* are,
and how they fit together.

**Prerequisites:** none. This is the place to start.

**What you need:** an internet connection — we read live examples from the public REF API.

## What is the REF?

The REF runs calculations against climate datasets, much like a CI/CD pipeline runs tests
against code. As new climate model output is published, the REF evaluates it against
reference data and produces figures and metrics — in near-real time.

The public deployment evaluates CMIP7 Assessment Fast Track data. Its results are served
from a website (<https://climate-ref.org>) and a public API (<https://api.climate-ref.org>).

Throughout these notebooks we talk to that API. The `ref_tutorials` helper package (shipped
with this repository) builds the API client for us:

In [ ]:
from ref_tutorials import get_client

client = get_client()
client

## Diagnostics

A **diagnostic** is a single, well-defined evaluation — for example "the global mean
surface temperature timeseries" or "the Atlantic overturning circulation strength".

Let's list the diagnostics the REF currently provides:

In [ ]:
from climate_ref_client.api.diagnostics import diagnostics_list

diagnostics = diagnostics_list.sync(client=client).data
print(f"{len(diagnostics)} diagnostics available\n")
for diagnostic in sorted(diagnostics, key=lambda d: d.name)[:10]:
    print(f"  {diagnostic.slug:35s} {diagnostic.name}")

Every diagnostic has a human-readable `name`, a short machine-friendly `slug`, and a
`description`. Let's look at one in full:

In [ ]:
example = diagnostics[0]
print("name:       ", example.name)
print("slug:       ", example.slug)
print("description:", example.description.strip())

## Providers

Diagnostics are grouped into **providers**. A provider is a package that knows how to
compute a family of diagnostics — examples include ESMValTool, the PCMDI Metrics Package
(PMP), and ILAMB. The REF orchestrates providers; it does not compute diagnostics itself.

Each diagnostic tells you which provider it belongs to:

In [ ]:
providers = {d.provider.slug: d.provider.name for d in diagnostics}
for slug, name in sorted(providers.items()):
    count = sum(1 for d in diagnostics if d.provider.slug == slug)
    print(f"  {slug:20s} {name}  ({count} diagnostics)")

## Datasets

A diagnostic needs **datasets** to run against: the climate model output under evaluation,
and the reference (often observational) data it is compared to. The REF tracks which
datasets are available and which combinations a diagnostic requires.

For the public API we do not handle raw datasets directly — the evaluations have already
been run. We will see locally fetched datasets in notebook 04.

## Executions

An **execution** is one run of a diagnostic against one specific group of datasets.
A single diagnostic is typically executed many times — once per model, experiment, or
scenario — so it has many executions, organised into *execution groups*.

Here are the execution groups of one diagnostic:

In [ ]:
with_executions = next(d for d in diagnostics if d.execution_groups)
print(f"Diagnostic: {with_executions.name}")
print(f"Execution groups: {len(with_executions.execution_groups)}")

## Metrics

An execution produces output. That output comes in two main shapes:

- **Metric values** — numbers that summarise a property of a model. A *scalar* is a single
  number (e.g. a root-mean-square error); a *series* is a number per index point
  (e.g. a seasonal cycle).
- **Files** — NetCDF data and figures for deeper analysis or custom plotting.

Notebook 02 shows how to retrieve metric values, and notebook 03 turns them into a
publication-ready figure.

## Recap

| Term | Meaning |
|------|---------|
| **Provider** | A package that computes a family of diagnostics (ESMValTool, PMP, ILAMB, ...) |
| **Diagnostic** | A single well-defined evaluation |
| **Dataset** | Climate model output or reference data a diagnostic runs against |
| **Execution** | One run of a diagnostic against one group of datasets |
| **Metric value** | A scalar or series result summarising a model |

**Next:** [02 — Querying the REF API](02-querying-the-api.ipynb).